## Configuration and Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import AutoTokenizer
from sklearn.feature_extraction.text import CountVectorizer
from tqdm import tqdm

# Enable progress_apply for pandas
tqdm.pandas()

# --- CONFIGURATION ---
MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
DATASET_PATH = "m-a-p/CodeFeedback-Filtered-Instruction"
DATASET_SPLIT = "train"
SAMPLE_SIZE = 15000
RANDOM_SEED = 42
SYSTEM_PROMPT = "You are a senior software engineer. Write clean, efficient, and well-documented code. Think through the logic step-by-step before providing the final code block."


## EDA

In [ ]:
def run_codefeedback_eda():
    print("1. Loading Dataset and Tokenizer...")
    dataset = load_dataset(DATASET_PATH, split=DATASET_SPLIT)
    df = dataset.to_pandas()

    print(f"Original dataset size: {len(df)} rows")

    print("\n2. Applying Python Filter & Extraction...")
    # Strict Python filtering mirroring the preprocessor
    df["lang"] = df["lang"].astype(str).str.strip().str.lower()
    df_python = df[df["lang"] == "python"].copy()

    print(f"Rows after filtering for Python: {len(df_python)}")

    # Sample the requested size (or max available)
    actual_sample_size = min(SAMPLE_SIZE, len(df_python))
    train_df = df_python.sample(n=actual_sample_size, random_state=RANDOM_SEED).copy()
    print(f"Sampled {actual_sample_size} rows for EDA.")

    # Apply pure code extraction
    train_df["pure_code"] = train_df["answer"]
    train_df = train_df.dropna(subset=["pure_code"]).copy()

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

    print("\n3. Calculating Token Length Distributions...")

    def calculate_lengths(row):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": str(row["query"]).strip()},
            {"role": "assistant", "content": str(row["pure_code"]).strip()},
        ]

        # Full text tokenization
        formatted_string = tokenizer.apply_chat_template(messages, tokenize=False)
        # Prompt only tokenization (System + User)
        prompt_only = messages[:2]
        formatted_prompt = tokenizer.apply_chat_template(prompt_only, tokenize=False)

        total_tokens = len(tokenizer.encode(formatted_string))
        prompt_tokens = len(tokenizer.encode(formatted_prompt))
        completion_tokens = total_tokens - prompt_tokens

        return pd.Series([prompt_tokens, completion_tokens, total_tokens])

    train_df[["prompt_tokens", "completion_tokens", "total_tokens"]] = (
        train_df.progress_apply(calculate_lengths, axis=1)
    )

    print("\nToken Length Percentiles:")
    print(
        train_df[["prompt_tokens", "completion_tokens", "total_tokens"]].describe(
            percentiles=[0.50, 0.90, 0.95, 0.99]
        )
    )

    # Calculate how many rows exceed the preprocessor's max_length of 512
    oversized = len(train_df[train_df["total_tokens"] > 512])
    print(
        f"\nWARNING: {oversized} rows ({oversized / len(train_df) * 100:.2f}%) exceed the 512 token limit."
    )

    print("\n4. Plotting Distributions...")
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    sns.histplot(
        train_df["prompt_tokens"], bins=50, ax=axes[0], color="blue"
    ).set_title("Prompt Token Length")
    sns.histplot(
        train_df["completion_tokens"], bins=50, ax=axes[1], color="orange"
    ).set_title("Completion Token Length")
    sns.histplot(
        train_df["total_tokens"], bins=50, ax=axes[2], color="green"
    ).set_title("Total Sequence Length")

    # Add vertical lines for the 512 threshold on the total tokens plot
    axes[2].axvline(
        512, color="red", linestyle="dashed", linewidth=2, label="512 Token Limit"
    )
    axes[2].legend()

    plt.tight_layout()
    plt.show()

    print("\n5. Vocabulary Frequency Analysis (Router Prep)...")
    vectorizer = CountVectorizer(stop_words="english", max_features=20)
    word_counts = vectorizer.fit_transform(train_df["query"]).sum(axis=0)
    words_freq = [
        (word, word_counts[0, idx]) for word, idx in vectorizer.vocabulary_.items()
    ]
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)

    words, counts = zip(*words_freq)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=list(counts), y=list(words), palette="viridis")
    plt.title("Top 20 Informative Words in CodeFeedback Queries")
    plt.show()

    print("\n6. Displaying 10 Standardized Examples...")
    print("=" * 80)
    # Reduced to 10 to avoid massive console spam, adjust if needed
    for i in range(10):
        row = train_df.iloc[i]
        print(f"\n--- EXAMPLE {i + 1} ---")
        print(f"TOTAL TOKENS: {row['total_tokens']}")
        print(f"QUERY:\n{row['query']}\n")
        print(f"STANDARDIZED RESPONSE:\n{row['pure_code']}")
        print("-" * 80)


## Execution

In [ ]:
if __name__ == "__main__":
    run_codefeedback_eda()
